In [1]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

In [18]:
path_to_all_preds = "/home/macraedc/rt_pred_results/MC Dropout/Xero_1/model_1/model_predictions.csv"

df = pd.read_csv(path_to_all_preds, sep=';', index_col=0)

df = df[df['Xerostomia_M06_true'] != -1]  # Filter out missing labels

df_val = df[df['Mode'] == 'val']
df_test = df[df['Mode'] == 'test']

val_preds = df_val['Xerostomia_M06_pred'].values
val_labels = df_val['Xerostomia_M06_true'].values

test_preds = df_test['Xerostomia_M06_pred'].values
test_labels = df_test['Xerostomia_M06_true'].values

In [23]:
import numpy as np
import torch

# ----------------------------
# Step 1: Predict on Calibration Set
# ----------------------------

calib_preds = val_preds
calib_targets = val_labels

# ----------------------------
# Step 2: Compute Nonconformity Scores (absolute errors)
# ----------------------------
nonconformity_scores = np.abs(calib_preds - calib_targets)

# Choose quantile for desired coverage (e.g., 90%)
alpha = 0.5
q = np.quantile(nonconformity_scores, 1 - alpha)

print(f"90% prediction interval radius: {q:.3f}")

# ----------------------------
# Step 3: Predict on Test Set and Apply Conformal Interval
# ----------------------------

test_intervals = []

with torch.no_grad():
    for y_pred in test_preds:
        
        # Build prediction interval [y - q, y + q], clipped to [0, 1]
        lower = np.maximum(0, y_pred - q)
        upper = np.minimum(1, y_pred + q)
        test_intervals.append((lower, upper))

# Example output for one test case:
for i in range(20):
    print(f"Prediction: {test_preds[i]:.3f}, 90% interval: [{test_intervals[i][0]:.3f}, {test_intervals[i][1]:.3f}]")


90% prediction interval radius: 0.404
Prediction: 0.291, 90% interval: [0.000, 0.695]
Prediction: 0.631, 90% interval: [0.227, 1.000]
Prediction: 0.112, 90% interval: [0.000, 0.516]
Prediction: 0.149, 90% interval: [0.000, 0.553]
Prediction: 0.430, 90% interval: [0.026, 0.834]
Prediction: 0.646, 90% interval: [0.242, 1.000]
Prediction: 0.371, 90% interval: [0.000, 0.775]
Prediction: 0.290, 90% interval: [0.000, 0.694]
Prediction: 0.508, 90% interval: [0.104, 0.912]
Prediction: 0.537, 90% interval: [0.132, 0.941]
Prediction: 0.363, 90% interval: [0.000, 0.767]
Prediction: 0.557, 90% interval: [0.153, 0.961]
Prediction: 0.255, 90% interval: [0.000, 0.659]
Prediction: 0.117, 90% interval: [0.000, 0.521]
Prediction: 0.091, 90% interval: [0.000, 0.496]
Prediction: 0.563, 90% interval: [0.159, 0.967]
Prediction: 0.504, 90% interval: [0.100, 0.908]
Prediction: 0.193, 90% interval: [0.000, 0.597]
Prediction: 0.556, 90% interval: [0.152, 0.960]
Prediction: 0.072, 90% interval: [0.000, 0.476]


In [42]:
def nonconformity_scores(probs, labels):
    return np.where(labels == 1, 1 - probs, probs)

# def fit_conformal_calibrator(model, X_calib, y_calib):
#     # Get model probabilities on calibration set
#     prob_pos = model.predict_proba(X_calib)[:, 1]
#     return nonconformity_scores(prob_pos, y_calib)

def predict_with_conformal(preds, calib_scores, alpha=0.1):
    # Predict probabilities for test set

    prob_pos = preds
    
    # Calculate prediction sets for each sample
    prediction_sets = []
    threshold = np.quantile(calib_scores, 1 - alpha)

    print(threshold)

    for p in prob_pos:
        set_ = []
        score_0 = p          # nonconformity score if label = 0
        score_1 = 1 - p      # nonconformity score if label = 1
        if score_0 <= threshold:
            set_.append(0)
        if score_1 <= threshold:
            set_.append(1)
        prediction_sets.append(set_)
    return prediction_sets




alpha = 0.05

# Step 1: Fit conformal calibrator
calib_scores = nonconformity_scores(val_preds, val_labels)

# Step 2: Predict conformal sets on test set
prediction_sets = predict_with_conformal(test_preds, calib_scores, alpha=alpha)

# Step 3: Evaluate
correct = 0
for true, pred_set in zip(test_labels, prediction_sets):
    if true in pred_set:
        correct += 1

coverage = correct / len(test_labels)
print(f"Conformal coverage (should be ≥ 1 - alpha = {1-alpha}): {coverage:.3f}")

# Optional: Check ambiguous predictions (uncertainty)
ambiguous = sum([len(s) == 2 for s in prediction_sets])
print(f"Ambiguous predictions (both labels included): {ambiguous}/{len(prediction_sets)}")


0.6858828579999999
Conformal coverage (should be ≥ 1 - alpha = 0.95): 0.907
Ambiguous predictions (both labels included): 156/257


In [48]:
prediction_sets

[[0],
 [0, 1],
 [0],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0],
 [0],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0],
 [0],
 [0, 1],
 [0],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0],
 [0],
 [0, 1],
 [0],
 [0],
 [0],
 [0],
 [1],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [1],
 [0],
 [0],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [0],
 [0, 1],
 [0],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [0, 1],
 [1],
 [0, 1],
 [0],
 [0, 1],
 [0, 1],
 [1],
 [0],
 [0, 1],
 [0, 1],
 